In [5]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
from optuna.samplers import TPESampler, GPSampler
import sys
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) 
import script_otim as script
import logging

In [ ]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "tpe_seed1" 
storage_name = "sqlite:///{}.db".format(study_name)

In [7]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "True"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 10108!


In [8]:
def objective(trial):
    a1 = trial.suggest_float("a1", 1.0, 1.0)
    a2 = trial.suggest_float("a2", 1.0, 1.0)
    n1 = trial.suggest_int("n1", 1, 100)
    n2 = trial.suggest_float("n2", 1.0, 10.0)
    n3 = trial.suggest_float("n3", 1.0, 10.0)
    n4 = trial.suggest_int("n4", 1, 100)
    perc = trial.suggest_float("perc", 0.2, 0.8)
    r = trial.suggest_int("r", 90, 150)

    df = script.makeAntenna(a1, a2, n1, n2, n3, n4, perc, r, temp_folder)
    df_filtrado = df[(df["Frequency [MHz]"] >= 0.3) & (df["Frequency [MHz]"] <= 1.5)]
    s11 = np.asarray(df_filtrado['|$S_{11}$| [dB]'], dtype=float)

    n = len(s11)
    idx1 = n // 3
    idx2 = 2 * n // 3

    weight = np.zeros(n)
    weight[:idx1] = 1
    weight[idx1:idx2] = 0.9
    weight[idx2:] = 0.7

    mask = s11 >= -10

    weighted_count = np.sum(mask * weight)

    weighted_fraction = weighted_count / np.sum(weight)

    return weighted_fraction

study = optuna.create_study(
    direction="minimize",    
    study_name=study_name, 
    storage=storage_name,
    sampler=TPESampler(seed=247),
    load_if_exists=True
)

#seed1 = 123
#seed2 = 173
#seed3 = 247

C:\Users\luis.costa\AppData\Local\Temp\ipykernel_15120\4035163326.py:36: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  sampler=GPSampler(seed=123),
[I 2026-05-04 20:39:49,814] A new study created in RDB with name: gp_seed1      


A new study created in RDB with name: gp_seed1      
A new study created in RDB with name: gp_seed1      


In [ ]:
study.optimize(objective, n_trials=500)

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 10108!
PyAEDT INFO: Project Project1623 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_H1I' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed E

[I 2026-05-04 21:23:15,402] Trial 0 finished with value: 0.9732569245463228 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 70, 'n2': 3.575254014553415, 'n3': 3.041663082077828, 'n4': 56, 'perc': 0.6316813818713379, 'r': 115}. Best is trial 0 with value: 0.9732569245463228.


Trial 0 finished with value: 0.9732569245463228 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 70, 'n2': 3.575254014553415, 'n3': 3.041663082077828, 'n4': 56, 'perc': 0.6316813818713379, 'r': 115}. Best is trial 0 with value: 0.9732569245463228.
Trial 0 finished with value: 0.9732569245463228 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 70, 'n2': 3.575254014553415, 'n3': 3.041663082077828, 'n4': 56, 'perc': 0.6316813818713379, 'r': 115}. Best is trial 0 with value: 0.9732569245463228.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 10108!
PyAEDT INFO: Project Project1624 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_DPP' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materi

[I 2026-05-04 22:03:57,112] Trial 1 finished with value: 1.0 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 99, 'n2': 7.1634676472637695, 'n3': 5.328387113359248, 'n4': 40, 'perc': 0.4059068096905217, 'r': 134}. Best is trial 0 with value: 0.9732569245463228.


Trial 1 finished with value: 1.0 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 99, 'n2': 7.1634676472637695, 'n3': 5.328387113359248, 'n4': 40, 'perc': 0.4059068096905217, 'r': 134}. Best is trial 0 with value: 0.9732569245463228.
Trial 1 finished with value: 1.0 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 99, 'n2': 7.1634676472637695, 'n3': 5.328387113359248, 'n4': 40, 'perc': 0.4059068096905217, 'r': 134}. Best is trial 0 with value: 0.9732569245463228.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 10108!
PyAEDT INFO: Project Project1625 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_GIA' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initializ

...

Trial 18 finished with value: 0.2989493791786056 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 41, 'n2': 4.652625350535939, 'n3': 9.999999999999998, 'n4': 1, 'perc': 0.6547624084481038, 'r': 135}. Best is trial 12 with value: 0.15663801337153774.
Trial 18 finished with value: 0.2989493791786056 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 41, 'n2': 4.652625350535939, 'n3': 9.999999999999998, 'n4': 1, 'perc': 0.6547624084481038, 'r': 135}. Best is trial 12 with value: 0.15663801337153774.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 10108!
PyAEDT INFO: Project Project1642 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_4N7' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Ma

[I 2026-05-05 06:43:20,408] Trial 19 finished with value: 0.7984718242597898 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 33, 'n2': 4.651353603665519, 'n3': 6.020654705564026, 'n4': 1, 'perc': 0.6581228291711478, 'r': 150}. Best is trial 12 with value: 0.15663801337153774.


Trial 19 finished with value: 0.7984718242597898 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 33, 'n2': 4.651353603665519, 'n3': 6.020654705564026, 'n4': 1, 'perc': 0.6581228291711478, 'r': 150}. Best is trial 12 with value: 0.15663801337153774.
Trial 19 finished with value: 0.7984718242597898 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 33, 'n2': 4.651353603665519, 'n3': 6.020654705564026, 'n4': 1, 'perc': 0.6581228291711478, 'r': 150}. Best is trial 12 with value: 0.15663801337153774.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 10108!
PyAEDT INFO: Project Project1643 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_5GT' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Ma

[I 2026-05-05 06:59:26,211] Trial 20 finished with value: 0.3724928366762178 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 4, 'n2': 4.472662117814684, 'n3': 9.446646046773244, 'n4': 1, 'perc': 0.657693598720781, 'r': 150}. Best is trial 12 with value: 0.15663801337153774.


Trial 20 finished with value: 0.3724928366762178 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 4, 'n2': 4.472662117814684, 'n3': 9.446646046773244, 'n4': 1, 'perc': 0.657693598720781, 'r': 150}. Best is trial 12 with value: 0.15663801337153774.
Trial 20 finished with value: 0.3724928366762178 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 4, 'n2': 4.472662117814684, 'n3': 9.446646046773244, 'n4': 1, 'perc': 0.657693598720781, 'r': 150}. Best is trial 12 with value: 0.15663801337153774.
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 10108!
PyAEDT INFO: Project Project1644 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_V72' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materi

In [ ]:
len(study.trials)

500

In [ ]:
study.best_params

{'a1': 1.0,
 'a2': 1.0,
 'n1': 55,
 'n2': 3.8612360075997505,
 'n3': 1.5051135486821772,
 'n4': 10,
 'perc': 0.5607382475229594,
 'r': 104}

In [ ]:
study.best_trial

FrozenTrial(number=304, state=TrialState.COMPLETE, values=[0.02865329512893983], datetime_start=datetime.datetime(2026, 4, 29, 17, 43, 8, 593891), datetime_complete=datetime.datetime(2026, 4, 29, 18, 11, 30, 104028), params={'a1': 1.0, 'a2': 1.0, 'n1': 55, 'n2': 3.8612360075997505, 'n3': 1.5051135486821772, 'n4': 10, 'perc': 0.5607382475229594, 'r': 104}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'a1': FloatDistribution(high=1.0, log=False, low=1.0, step=None), 'a2': FloatDistribution(high=1.0, log=False, low=1.0, step=None), 'n1': IntDistribution(high=100, log=False, low=1, step=1), 'n2': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'n3': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'n4': IntDistribution(high=100, log=False, low=1, step=1), 'perc': FloatDistribution(high=0.8, log=False, low=0.2, step=None), 'r': IntDistribution(high=150, log=False, low=90, step=1)}, trial_id=305, value=None)

In [ ]:
d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True